# 06 Hybrid architecture: deterministic query tools + LLM router, scored with the team benchmark

**Date:** 2026-10-22, updated 2026-11-17
**Author:** Xincheng (George) Ding
**Needs network** when `IATI_LIVE_EVAL=1` (Aura + the models listed in `.env`). Default is a dry run.

**Why.** On benchmark v1, notebook 05's free Text2Cypher agent scored 92 to 98%, with the remaining errors in
exactly the places a client cannot accept (a financial total). The SQL baseline is 100% because a human wrote each query.
This notebook makes that guarantee part of the product: every recurring question type becomes a **parameterised Cypher
template** (`fa26/src/tools/templates.py`, 21 templates incl. top funders, shared partners, funding over time, counts),
Alex's five retrieval tools join through `src/tools/adapter.py`, and an **entity resolver** (`resolver.py`) maps names to
identifiers. The LLM only chooses the tool and its arguments; it writes Cypher only when no tool fits. A wrongly chosen
tool can give the wrong answer; a chosen tool computes the number itself.

**Configurations.**

| config | who routes | who writes Cypher | runs on |
| --- | --- | --- | --- |
| A `free_cypher` | LLM | LLM | notebook 05 (scored there) |
| B `templates_manual` | a human maps question to tool (ceiling, like the SQL baseline) | templates | benchmark v1 only (question-id routes) |
| C `hybrid_llm_router` | LLM picks tool + args, falls back to free Cypher | templates, LLM as fallback | v1 and every v2 split |

Benchmark: v1 (20 questions, `benchmark/score.py`) with no bundle, or one v2 split (100 questions in `regression` 20 /
`development` 40 / `heldout` 40, `benchmark/v2/score_v2.py`) with `IATI_BENCHMARK_BUNDLE`. The saved outputs below are
the v2 `heldout` run (40 questions, config C, Sonnet 4.6 and GLM-5.3): the first and only heldout run, frozen.

**Outputs:** submissions in `fa26/data/processed/06_submission_<config>_<model>_<run>.json`, scores in
`fa26/benchmark/results/live_<run>_hybrid/` (v1) or `fa26/benchmark/v2/results/live/06_<split>_<run>/` (v2),
long table `06_hybrid_eval_<run>.csv`.

## V2 inputs and offline verification
Benchmark v2 has 100 questions in three splits (`regression` 20, `development` 40, `heldout` 40). Export the public bundle of one split, then set `IATI_BENCHMARK_BUNDLE` to its absolute directory before running. Tune only on `development`; the first `heldout` run is frozen. With no bundle, the original v1 questions are used. `IATI_LIVE_EVAL=0` runs fake models without a database; set `1` only for an authorized live run. V2 scoring runs after generation in a separate evaluator process. The bundle supplies question ids and text only, no evaluator contracts or expected value types. This notebook skips its hard-coded v1 manual routes for v2 and evaluates the LLM router plus fallback. No templates were added based on heldout failures. Results include benchmark and split metadata; comparisons use only submissions from this run.

In [1]:
from pathlib import Path
import os, sys, re, json, time, subprocess, glob
from decimal import Decimal
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "fa26" / "benchmark" / "manifest.json").exists())
FA26 = ROOT / "fa26"
sys.path.insert(0, str(FA26 / "benchmark")); sys.path.insert(0, str(FA26 / "src"))
from common import verify_snapshot, load_json, save_json
from text2cypher_eval import guard_query
from tools import TOOLS, run_tool, validate_args, tool_catalog, OrgResolver
from tools.adapter import run_any, catalog_all, ToolClarification   # Alex's retrieval tools (rt_*) behind the same interface
from tools.contract import evidence_error, clean_evidence, normalise_facts, single_v2_reason, V2_REASON_GUIDE   # output contract (evidence ids, scalar keys, reasons)
SNAPSHOT_ID = verify_snapshot()["snapshot_id"]

import pandas as pd
from datetime import datetime, timezone
from dotenv import load_dotenv
load_dotenv(FA26 / ".env", override=False)
DRY_RUN = os.environ.get("IATI_LIVE_EVAL", "0") != "1"
RUN_ID = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H%M%S_%f")
PROCESSED = FA26 / "data" / "processed"; PROCESSED.mkdir(parents=True, exist_ok=True)
RESULTS = FA26 / "benchmark" / "results" / f"live_{RUN_ID}_hybrid"
QUESTIONS = load_json(FA26 / "benchmark" / "questions.json")
REASON_CODES = ["NEEDS_DATA_MISSING", "HISTORY_UNAVAILABLE", "FUNDING_DEFINITION_REQUIRED", "CURRENCY_BASIS_REQUIRED", "PERIOD_REQUIRED", "RAW_XML_UNAVAILABLE"]
print("run", RUN_ID, "| dry run:", DRY_RUN, "| tools:", len(TOOLS))
from benchmark_bundle import load_bundle, submission, score_command, read_scores
BUNDLE = load_bundle(os.environ["IATI_BENCHMARK_BUNDLE"], SNAPSHOT_ID) if os.environ.get("IATI_BENCHMARK_BUNDLE") else None
if BUNDLE:
    RESULTS = FA26 / "benchmark" / "v2" / "results" / "live" / ("06_" + BUNDLE["template"]["split"] + "_" + RUN_ID)
    QUESTIONS = BUNDLE["questions"]
    REASON_CODES = BUNDLE["reasons"]


run 2026-11-05_204405_996398 | dry run: False | tools: 21


## 1. The tool layer (21 templates on the team schema `retrieval/schema_prototype.py`, plus Alex's 5 retrieval tools through `src/tools/adapter.py`)

In [2]:
pd.set_option("display.max_colwidth", 120)
pd.DataFrame([{"tool": n, "params": ", ".join(t["params"]) or "-", "description": t["description"][:110]} for n, t in TOOLS.items()])

,tool,params,description
0,activities_per_publisher,"publisher, start_from, start_to","Number of distinct activities per publisher, optionally restricted to activities whose start date (start_actua"
1,activities_per_publisher_by_country,country_code,Number of distinct activities per publisher that target one recipient country (ISO-2 code). Only publishers wi
2,hotspot_countries_per_publisher,-,Number of distinct hunger-hotspot countries each publisher targets (TARGETS with is_hotspot = true).
3,activities_by_match_basis,-,Number of distinct activities grouped by how the hunger filter admitted them (match_basis: 'sector' or 'keywor
4,count_activities_by_sector,"publisher, sector_key",Scalar count of distinct activities of one publisher classified with one activity-level sector key (e.g. '1:72
5,latest_update_per_publisher,-,Latest publisher-reported last_updated calendar date (YYYY-MM-DD) per publisher.
6,activities_with_location_per_publisher,-,Number of distinct activities per publisher that have at least one Location node.
7,activities_per_sdg,-,Number of distinct activities linked to each SDG key (e.g. '2:2').
8,participants_by_role,-,"Number of distinct organisation refs occurring in each participation role code ('1' funding, '2' accountable,"
9,multi_publisher_participants,min_publishers,"Organisation refs that participate in activities of at least min_publishers distinct publishers, with the numb"


In [3]:
resolver = OrgResolver.from_csv(FA26 / "benchmark" / "snapshot" / "nodes_organisations.csv")
for q in ["UK government", "WFP", "ECHO", "DRC", "Oxfam Novib", "Multilateral"]:
    r = resolver.resolve(q); print(f"{q:<14} -> {r['kind']:<12} {r['refs'][:4]}   {r['note'][:70]}")
print("\nduplicate-looking identifier groups (merge list candidates):")
print(pd.DataFrame(resolver.duplicate_groups()).to_string(index=False))

UK government  -> organisation ['XM-DAC-12', 'XM-DAC-12-1']   2 identifiers represent UK_GOV; include all of them unless the questio
WFP            -> organisation ['name:UN World Food Programme (WFP)', 'XM-DAC-41140']   2 identifiers represent WFP; include all of them unless the question n
ECHO           -> organisation ['name:EU - ECHO (EC Humanitarian Aid Dept)', 'XI-IATI-EC_ECHO']   2 identifiers represent EU_ECHO; include all of them unless the questi
DRC            -> country      ['CD']   Country code for TARGETS / recipient filters, not an organisation.
Oxfam Novib    -> organisation ['NL-KVK-27108436']   Exact organisation name.
Multilateral   -> organisation []   Pooled-fund placeholder ('Multilateral'), not an organisation; no iden

duplicate-looking identifier groups (merge list candidates):
       group  n                                                                                                                                                                          

## 2. Read-only executor with parameters

In [4]:
if DRY_RUN:
    def run_cypher(q, params=None): return []
    print("dry run: no database")
else:
    from neo4j import GraphDatabase, Query, READ_ACCESS
    driver = GraphDatabase.driver(os.environ["NEO4J_URI"], auth=(os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"]))
    driver.verify_connectivity()
    DB = os.environ.get("NEO4J_DATABASE", "neo4j")
    def run_cypher(q, params=None):
        guard_query(q)
        with driver.session(database=DB, default_access_mode=READ_ACCESS) as s:
            return [dict(r) for r in s.run(Query(q, timeout=60), **(params or {}))]
    print("Neo4j connected; nodes:", run_cypher("MATCH (n) RETURN count(n) AS c")[0]["c"])

Neo4j connected; nodes: 27280


## 3. Shared formatting (same conventions as notebook 05)

In [5]:
def fmt_value(v, kind):
    if kind == "integer": return str(int(round(float(v))))
    if kind == "decimal": return str(Decimal(str(v)).quantize(Decimal("0.01")))
    if kind == "date": return str(v)[:10]
    if isinstance(v, float) and not v.is_integer(): return str(Decimal(str(v)).quantize(Decimal("0.01")))
    if isinstance(v, (int, float)): return str(int(v))
    return str(v)

def rows_to_facts(rows, kind):
    return {str(r["key"]): {"value": fmt_value(r["value"], kind), "evidence": sorted({str(e) for e in (r.get("evidence") or []) if e})} for r in rows}

def make_answer(q, status, facts=None, reasons=None, explanation=None, latency_ms=None):
    a = {"id": q["id"], "status": status, "facts": facts or {} if status == "answer" else {},
         "reasons": reasons or [] if status != "answer" else [], "latency_ms": latency_ms}
    if explanation: a["explanation"] = explanation
    return a

## 4. Config B: templates with manual routing (the ceiling)

A human maps each benchmark question to a tool call. If this scores below 100%, the template is wrong, not the model.
The four limitation questions use fixed policy rules, as the SQL baseline does.

In [6]:
ROUTE = {
    "Q01": ("activities_per_publisher", {}),
    "Q02": ("hotspot_countries_per_publisher", {}),
    "Q03": ("activities_per_publisher_by_country", {"country_code": "SO"}),
    "Q04": ("multi_publisher_participants", {"min_publishers": 2}),
    "Q05": ("participants_by_role", {}),
    "Q06": ("activities_by_match_basis", {}),
    "Q07": ("count_activities_by_sector", {"publisher": "wfp", "sector_key": "1:72040"}),
    "Q08": ("activities_per_publisher", {"start_from": "2020-01-01", "start_to": "2026-01-01"}),
    "Q09": ("latest_update_per_publisher", {}),
    "Q10": ("transaction_total", {"publisher": "wfp", "type_code": "1", "currency": "USD", "date_from": "2020-01-01", "date_to": "2021-01-01"}),
    "Q11": ("transaction_total", {"publisher": "oxfamgb", "type_code": "3", "currency": "GBP", "date_from": "2020-01-01", "date_to": "2021-01-01"}),
    "Q12": ("transaction_total", {"publisher": "oxfamus", "type_code": "4", "currency": "USD"}),
    "Q13": ("negative_transactions_per_publisher", {}),
    "Q14": ("budget_total", {"type_code": "1", "status_code": "2", "currency": "USD"}),
    "Q15": ("activities_per_sdg", {}),
    "Q16": ("activities_with_location_per_publisher", {}),
}
POLICY = {  # limitation questions: the snapshot cannot support these; same rules as the SQL baseline
    "Q17": ("abstain", ["NEEDS_DATA_MISSING"]), "Q18": ("abstain", ["HISTORY_UNAVAILABLE"]),
    "Q19": ("clarify", ["FUNDING_DEFINITION_REQUIRED", "CURRENCY_BASIS_REQUIRED", "PERIOD_REQUIRED"]), "Q20": ("abstain", ["RAW_XML_UNAVAILABLE"]),
}
records, submissions = [], []
if not BUNDLE:
    answers = []
    for q in QUESTIONS:
        t0 = time.time()
        if q["id"] in ROUTE:
            name, args = ROUTE[q["id"]]
            res = run_tool(run_cypher, name, args)
            ans = make_answer(q, "answer", rows_to_facts(res["rows"], q.get("value_type", "auto")), latency_ms=round((time.time() - t0) * 1000, 1))
            records.append({"run": RUN_ID, "config": "templates_manual", "model": "none", "qid": q["id"], "status": "answer", "tool": name,
                            "args": json.dumps(args), "n_facts": len(ans["facts"]), "error": None, "latency_ms": ans["latency_ms"], "cypher": res["cypher"]})
        else:
            status, reasons = POLICY[q["id"]]
            ans = make_answer(q, status, reasons=reasons, explanation="policy rule", latency_ms=round((time.time() - t0) * 1000, 1))
            records.append({"run": RUN_ID, "config": "templates_manual", "model": "none", "qid": q["id"], "status": status, "tool": "policy",
                            "args": json.dumps(reasons), "n_facts": 0, "error": None, "latency_ms": ans["latency_ms"], "cypher": None})
        answers.append(ans)
        print(f"{q['id']} {ans['status']:<8} facts={len(ans['facts'])}  {records[-1]['tool']}")
    p = PROCESSED / f"06_submission_templates_manual_{RUN_ID}.json"
    save_json(p, {"run_name": f"templates_manual__neo4j_templates__{RUN_ID}", "snapshot_id": SNAPSHOT_ID, "answers": answers})
    submissions.append(p); print("wrote", p.name)
else:
    print("V2: manual v1 question-ID baseline skipped; run the LLM router below.")


V2: manual v1 question-ID baseline skipped; run the LLM router below.


## 5. Config C: hybrid agent (LLM router over tools, free Cypher as fallback) in LangGraph

`plan` asks the model for one JSON object: a status (answer / abstain / clarify), and for an answer either a tool name with
arguments or `free_cypher` with a query. `validate_args` rejects anything outside the tool's parameter schema, so the model
can only choose, never inject. Invalid arguments or a failed execution are fed back for up to 2 retries. The free-Cypher
fallback reuses the guard → execute → validate path from notebook 05.

In [7]:
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, END
from langchain_core.messages import SystemMessage, HumanMessage

SCHEMA_SHORT = '''Graph (loaded by fa26/retrieval/load_neo4j.py): (:Organisation {ref,name,publisher,source_id})-[:REPORTS]->(:Activity) and
(:Organisation)-[:PARTICIPATES_IN {role '1' funding|'2' accountable|'3' extending|'4' implementing, role_name, source_ids}]->(:Activity);
(:Activity {iati_identifier,publisher,title,status '2' implementation|'3' finalisation(completed),status_name,humanitarian '1'|'0'|null,default_currency,match_basis,
 start_actual,start_planned,end_actual,end_planned (Date or null), start_date,end_date (Date: actual else planned), last_updated (DateTime), hotspot_countries (list), source_id});
Activity-[:TARGETS {is_hotspot boolean, source_ids}]->(:Country {code,name}); Activity-[:CLASSIFIED_AS {source_ids}]->(:Sector {sector_key}); Activity-[:LOCATED_AT {source_ids}]->(:Location {location_id,name});
Activity-[:CONTRIBUTES_TO {source_ids}]->(:SDG {sdg_key}); Activity-[:HAS_TRANSACTION {source_ids}]->(:Transaction {tx_id,type '1'..'4',type_name,date (Date or null),value (float, signed),currency,source_id});
(:Organisation)-[:PROVIDES {source_ids}]->(:Transaction)-[:RECEIVED_BY {source_ids}]->(:Organisation); Activity-[:HAS_BUDGET {source_ids}]->(:Budget {type,status,value (float),currency,period_start,period_end,source_id}).
Publisher slugs: oxfamgb, oxfamus, wfp. Transaction type codes: 1 incoming funds, 2 outgoing commitment, 3 disbursement, 4 expenditure. Empty values are null (never ''); compare dates with date('YYYY-MM-DD'), e.g. a.start_actual < date('2020-01-01'); t.date.year gives the year.
EVIDENCE in free_cypher: nodes have source_id, relationships have source_ids (a list); return collect(DISTINCT a.source_id) + reduce(acc=[], l IN collect(DISTINCT r.source_ids) | acc + coalesce(l, [])) AS evidence. Never cite iati_identifier, ref or location_id.
Negative transactions are their own category: 'amount' sums value > 0, 'adjustments' sums value < 0, 'net' sums both (use the measure parameter of money tools).'''

SYSTEM = '''You route questions about one fixed snapshot of IATI humanitarian aid data to deterministic query tools.
Reply with ONE JSON object only:
{"status": "answer" | "abstain" | "clarify",
 "reasons": [],                       // abstain/clarify only, from ''' + json.dumps(REASON_CODES) + '''
 "explanation": "one sentence",
 "tool": "<tool name>" | "free_cypher" | null,
 "args": {...},                       // the tool's parameters, exactly as declared; omit optional ones you do not need
 "cypher": ""}                        // only when tool == "free_cypher": one read-only query returning key, value, evidence

Prefer a tool whenever one matches the question's contract exactly; use free_cypher only when no tool fits.
For "how many activities ..." with any combination of conditions use count_activities (set only the filters the question states);
for counts of transactions, organisations or locations use count_transactions / count_organisations / count_locations; for earliest/latest dates use date_extreme.
Single-value answers must be keyed exactly: count (a count), total (a monetary amount), date (a date); per-currency amounts use currency codes as keys.
Abstain for needs-based rankings, historical versions or original XML fields; clarify when a funding question does not fix the
financial measure, currency basis and period; in that case list EVERY missing dimension in reasons (FUNDING_DEFINITION_REQUIRED, CURRENCY_BASIS_REQUIRED, PERIOD_REQUIRED). Country names must be converted to ISO-2 codes (Somalia = SO).

TOOLS:
''' + catalog_all() + "\n\n" + SCHEMA_SHORT

if BUNDLE:
    SYSTEM += "\nPUBLIC BUNDLE INSTRUCTIONS:\n" + BUNDLE["instructions"] + "\n\n" + V2_REASON_GUIDE

class S(TypedDict, total=False):
    qid: str; question: str; contract: str; value_type: str
    raw: str; status: str; reasons: list; explanation: str; tool: Optional[str]; args: dict; cypher: str
    error: Optional[str]; retries: int; rows: list; facts: dict; model_calls: int

def parse_plan(text):
    m = re.search(r"\{.*\}", text, re.S); obj = json.loads(m.group(0)) if m else {}
    status = obj.get("status", "answer"); status = status if status in ("answer", "abstain", "clarify") else "answer"
    return status, [r for r in obj.get("reasons", []) if r in REASON_CODES], obj.get("explanation", ""), obj.get("tool"), obj.get("args") or {}, (obj.get("cypher") or "").strip().strip("`").rstrip(";")

def build_hybrid(llm, run_cypher):
    def plan(st):
        human = f"Question {st['qid']}: {st['question']}\nContract: {st['contract']}\nValue type: {st['value_type']}"
        if st.get("error"):
            human += f"\n\nYour previous attempt failed: {st['error']}\nPrevious choice: tool={st.get('tool')} args={json.dumps(st.get('args'))}\nReturn a corrected JSON object."
        out = llm.invoke([SystemMessage(content=SYSTEM), HumanMessage(content=human)])
        text = out.content if isinstance(out.content, str) else "\n".join(b.get("text", "") for b in out.content if isinstance(b, dict))
        try:
            status, reasons, expl, tool, args, cypher = parse_plan(text)
            if st["qid"].startswith("V2Q"): reasons = single_v2_reason(reasons, status)   # v2: exactly one reason code
            err = None if status != "answer" or tool else "no tool chosen"
            return {"raw": text, "status": status, "reasons": reasons, "explanation": expl, "tool": tool, "args": args, "cypher": cypher,
                    "error": err, "rows": [], "model_calls": st.get("model_calls", 0) + 1}
        except Exception as e:
            return {"raw": text, "status": "answer", "tool": None, "error": f"unparseable JSON: {type(e).__name__}", "rows": [], "model_calls": st.get("model_calls", 0) + 1}

    def run_template(st):
        try:
            res = run_any(run_cypher, st["tool"], st["args"])
            return {"rows": res["rows"], "cypher": res["cypher"], "args": res["params"], "error": None if res["rows"] or st["value_type"] == "decimal" else "tool returned no rows"}
        except ToolClarification as e:   # ambiguous organisation/country: ask instead of guessing
            ids = [str(c.get("id", c)) for c in e.candidates]
            scope = "country_scope" if ids and all(len(i) == 2 and i.isupper() for i in ids) else "organisation_scope"   # benchmark v2 reason vocabulary
            return {"rows": [], "status": "clarify", "reasons": [scope], "error": None, "explanation": f"{e}; candidates: " + ", ".join(ids[:5])}
        except Exception as e:
            return {"rows": [], "error": f"{type(e).__name__}: {str(e)[:300]}"}

    def run_free(st):
        try:
            guard_query(st["cypher"]); rows = run_cypher(st["cypher"])
        except Exception as e:
            return {"rows": [], "error": f"free_cypher failed {type(e).__name__}: {str(e)[:300]}"}
        for r in rows:
            if set(r) != {"key", "value", "evidence"}: return {"rows": [], "error": f"free_cypher must return key, value, evidence; got {sorted(r)}"}
        ev_err = evidence_error(rows)          # output contract: evidence must be source ids, else retry with the reason
        if ev_err: return {"rows": [], "error": ev_err}
        return {"rows": rows, "error": None if rows or st["value_type"] == "decimal" else "free_cypher returned no rows"}

    def format_(st):
        if st["status"] != "answer" or st.get("error"): return {"facts": {}}
        rows = st.get("rows") or [{"key": "total", "value": 0, "evidence": []}]
        for r in rows: r["evidence"] = clean_evidence(r.get("evidence"))[0]      # never hand the scorer an invalid id
        facts = rows_to_facts(rows, st["value_type"])
        return {"facts": normalise_facts(facts, st["question"], legacy=st["qid"].startswith("Q"))}   # scalar key = count|total|date

    def after_plan(st):
        if st["status"] != "answer": return "format"
        if st.get("error"): return "retry" if st.get("retries", 0) < 2 else "format"
        return "free" if st["tool"] == "free_cypher" else "template"
    def after_exec(st): return ("retry" if st.get("retries", 0) < 2 else "format") if st.get("error") else "format"

    g = StateGraph(S)
    for n, f in [("plan", plan), ("template", run_template), ("free", run_free), ("format", format_), ("retry", lambda st: {"retries": st.get("retries", 0) + 1})]:
        g.add_node(n, f)
    g.set_entry_point("plan")
    g.add_conditional_edges("plan", after_plan, {"format": "format", "retry": "retry", "free": "free", "template": "template"})
    g.add_conditional_edges("template", after_exec, {"retry": "retry", "format": "format"})
    g.add_conditional_edges("free", after_exec, {"retry": "retry", "format": "format"})
    g.add_edge("retry", "plan"); g.add_edge("format", END)
    return g.compile()

In [8]:
# ---- model switch: edit this line, never .env. None = every model registered in .env.
# Command line override without editing anything: IATI_RUN_MODELS="anthropic/claude-sonnet-4-6,zai/glm-5.3"
RUN_MODELS = ["anthropic/claude-sonnet-4-6", "zai/glm-5.3"]

def env(k, default=None): return os.environ.get(k, default) or default
MODELS = []
if env("ANTHROPIC_API_KEY"):
    for m in (env("ANTHROPIC_EVAL_MODELS", "") or "").split(","):
        if m.strip(): MODELS.append({"name": f"anthropic/{m.strip()}", "provider": "anthropic", "model": m.strip()})
for prov in ("OPENAI", "GROQ", "NVIDIA", "ZAI", "DASHSCOPE", "DEEPSEEK", "MOONSHOT", "OPENROUTER"):
    if not env(f"{prov}_API_KEY"): continue
    url = env(f"{prov}_BASE_URL")
    if prov != "OPENAI" and not (url and url.startswith("https://")): continue
    for k, v in sorted(os.environ.items()):
        if (k == f"{prov}_MODEL" or k.startswith(f"{prov}_MODEL_")) and v.strip():
            MODELS.append({"name": f"{prov.lower()}/{v.strip()}", "provider": "openai_compat", "model": v.strip(), "key_env": f"{prov}_API_KEY", "base_url": url})
if DRY_RUN: MODELS = [{"name": "fake/router", "provider": "fake"}]
assert MODELS, "No models registered in fa26/.env"
_sel = [m.strip() for m in os.environ.get("IATI_RUN_MODELS", "").split(",") if m.strip()] or RUN_MODELS
if _sel and not DRY_RUN:
    _missing = sorted(set(_sel) - {m["name"] for m in MODELS}); assert not _missing, f"RUN_MODELS not registered in .env: {_missing}"
    MODELS = [m for m in MODELS if m["name"] in _sel]

def make_llm(spec):
    if spec["provider"] == "anthropic":
        from langchain_anthropic import ChatAnthropic
        return ChatAnthropic(model=spec["model"], temperature=0, max_tokens=1200, timeout=120)
    if spec["provider"] == "openai_compat":
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=spec["model"], api_key=os.environ[spec["key_env"]], base_url=spec["base_url"], temperature=0, max_tokens=1200, timeout=180)
    from langchain_core.language_models.fake_chat_models import FakeListChatModel
    canned = [json.dumps({"status": "answer", "tool": "activities_per_publisher", "args": {}, "explanation": "dry"})] * 16 + \
             [json.dumps({"status": "abstain", "reasons": ["NEEDS_DATA_MISSING"], "tool": None, "explanation": "dry"})] * 4
    return FakeListChatModel(responses=canned * 3)
pd.DataFrame(MODELS)[["name", "provider"]]

,name,provider
0,anthropic/claude-sonnet-4-6,anthropic
1,zai/glm-5.3,openai_compat


In [9]:
for spec in MODELS:
    agent = build_hybrid(make_llm(spec), run_cypher)
    answers = []
    for q in QUESTIONS:
        t0 = time.time()
        try:
            st = agent.invoke({"qid": q["id"], "question": q["question"], "contract": q.get("contract", ""), "value_type": q.get("value_type", "auto"), "retries": 0, "model_calls": 0})
        except Exception as e:
            st = {"status": "answer", "facts": {}, "reasons": [], "error": f"{type(e).__name__}: {str(e)[:200]}", "retries": 0}
        lat = round((time.time() - t0) * 1000, 1)
        ans = make_answer(q, st.get("status", "answer"), st.get("facts", {}), st.get("reasons", []), st.get("explanation"), lat)
        answers.append(ans)
        records.append({"run": RUN_ID, "config": "hybrid_llm_router", "model": spec["name"], "qid": q["id"], "status": ans["status"], "tool": st.get("tool"),
                        "args": json.dumps(st.get("args")), "n_facts": len(ans["facts"]), "error": st.get("error"), "retries": st.get("retries", 0),
                        "model_calls": st.get("model_calls", 0), "latency_ms": lat, "cypher": st.get("cypher")})
        print(f"{spec['name']:<26} {q['id']} {ans['status']:<8} tool={str(st.get('tool')):<40} facts={len(ans['facts']):<3} retries={st.get('retries',0)} {lat:>7.0f} ms {('ERR ' + st['error'][:60]) if st.get('error') else ''}")
    p = PROCESSED / f"06_submission_hybrid_{spec['name'].replace('/', '_')}_{RUN_ID}.json"
    save_json(p, submission(BUNDLE, f"hybrid_llm_router__{spec['name'].replace('/', '_')}__{RUN_ID}", SNAPSHOT_ID, answers, DRY_RUN))
    submissions.append(p); print("wrote", p.name)
results = pd.DataFrame(records); results.to_csv(PROCESSED / f"06_hybrid_eval_{RUN_ID}.csv", index=False)
results.groupby(["config", "model"]).agg(n=("qid", "count"), errors=("error", lambda s: s.notna().sum()), latency_ms=("latency_ms", "mean")).round(0)

anthropic/claude-sonnet-4-6 V2Q011 answer   tool=count_activities                         facts=1   retries=0    3359 ms 


anthropic/claude-sonnet-4-6 V2Q012 answer   tool=date_extreme                             facts=1   retries=0    2355 ms 


anthropic/claude-sonnet-4-6 V2Q013 answer   tool=date_extreme                             facts=1   retries=0    2038 ms 


anthropic/claude-sonnet-4-6 V2Q014 abstain  tool=None                                     facts=0   retries=0    5196 ms 


anthropic/claude-sonnet-4-6 V2Q015 answer   tool=count_transactions                       facts=1   retries=1    5100 ms 


anthropic/claude-sonnet-4-6 V2Q016 answer   tool=free_cypher                              facts=1   retries=1    6770 ms 


... (74 outputs left out)

zai/glm-5.3                V2Q078 abstain  tool=None                                     facts=0   retries=0   27645 ms 


zai/glm-5.3                V2Q079 abstain  tool=None                                     facts=0   retries=0    9151 ms 


zai/glm-5.3                V2Q080 abstain  tool=None                                     facts=0   retries=0    2427 ms 
wrote 06_submission_hybrid_zai_glm-5.3_2026-11-05_204405_996398.json


n  errors  latency_ms
config            model                                              
hybrid_llm_router anthropic/claude-sonnet-4-6  40       5      6026.0
                  zai/glm-5.3                  40       2     49434.0

## 6. Score this run (only this notebook's submissions; notebook 05 is scored in notebook 05)

In [10]:
cmd = score_command(sys.executable, FA26, submissions, RESULTS, BUNDLE)
proc = subprocess.run(cmd, capture_output=True, text=True, cwd=ROOT)
print(proc.stdout); print(proc.stderr[-2000:] if proc.returncode else "")
assert proc.returncode == 0, "scorer rejected a submission"

[
  {
    "run_name": "hybrid_llm_router__anthropic_claude-sonnet-4-6__2026-11-05_204405_996398",
    "synthetic": false,
    "summary": {
      "questions": 40,
      "correctness_pct": 75.0,
      "evidence_f1_pct": 61.85,
      "behavior_pct": 87.5,
      "overall_pct": 72.59,
      "latency_samples": 40,
      "mean_latency_ms": 6025.98
    },
    "cost_usd": null,
    "by_category": {
      "direct": {
        "questions": 10,
        "correctness_pct": 80.0,
        "evidence_f1_pct": 69.22,
        "behavior_pct": null,
        "overall_pct": 76.77,
        "latency_samples": 10,
        "mean_latency_ms": 4596.5199999999995
      },
      "financial": {
        "questions": 8,
        "correctness_pct": 62.5,
        "evidence_f1_pct": 54.17,
        "behavior_pct": null,
        "overall_pct": 60.0,
        "latency_samples": 8,
        "mean_latency_ms": 9164.5125
      },
      "limitations": {
        "questions": 5,
        "correctness_pct": 80.0,
        "evidence_f1_pct

In [11]:
details = read_scores(RESULTS, BUNDLE)
rows = [{"run": r["summary"]["run_name"], "config": r["summary"]["run_name"].split("__")[0], "qid": q["id"], "score": round(q["score"], 2)}
        for r in details["runs"] for q in r["questions"]]
per_q = pd.DataFrame(rows)
summary = pd.DataFrame([r["summary"] for r in details["runs"]])[["run_name", "correctness_pct", "evidence_f1_pct", "behavior_pct", "overall_pct", "mean_latency_ms"]]
display(summary)
display(per_q.pivot(index="qid", columns="run", values="score"))

,run_name,correctness_pct,evidence_f1_pct,behavior_pct,overall_pct,mean_latency_ms
0,hybrid_llm_router__anthropic_claude-sonnet-4-6__2026-11-05_204405_996398,75.0,61.85,87.5,72.59,6025.9800
1,hybrid_llm_router__zai_glm-5.3__2026-11-05_204405_996398,90.0,83.09,100.0,88.94,49434.3725


run,hybrid_llm_router__anthropic_claude-sonnet-4-6__2026-11-05_204405_996398,hybrid_llm_router__zai_glm-5.3__2026-11-05_204405_996398
qid,,
V2Q011,1.00,1.00
V2Q012,1.00,1.00
V2Q013,1.00,1.00
V2Q014,0.00,0.00
V2Q015,1.00,1.00
V2Q016,0.98,0.98
V2Q017,0.70,0.70
V2Q018,1.00,1.00
V2Q019,0.00,0.00


## 7. How to read it

- **templates_manual below 100%** (v1 only) means a template disagrees with the benchmark contract; fix the template (the Cypher is in
  `src/tools/templates.py`) and rerun this section only. This is the one place where a human still has to be right once.
- **hybrid above free_cypher** on the money questions (v1 Q10 to Q14) is the product argument: the model picked the tool and
  the tool computed the number. Where hybrid is below, look at the `tool` and `args` columns in the CSV: the failure is a
  routing failure (wrong tool or wrong argument), which is fixable with examples, not a Cypher failure.
- **Evidence F1** for templates should be close to 100% because each template cites the same rows the benchmark reference
  uses (relationship rows plus the activities they join through). Shortfalls here are template design, not model behaviour.
- **Retries** are visible per question in the CSV (`retries`, `model_calls`): a failed output check sends the error back to the planner, at most twice.
- Next: give the resolver's identifier groups to the pipeline team as the merge list. The same router is packaged as `src/ask.py` for the Ask page; it has its own system prompt and has not been scored on the benchmark itself.